# Dobre praktyki w Data Science z asystentem AI (GitHub Copilot)

**Plan notatnika:**
1. Rola asystenta AI w pracy analityka
2. Jak pisać prompty w projektach Data Science
3. Typowe pułapki kodu wygenerowanego przez AI (z przykładami)
4. Weryfikacja: lista kontrolna
5. Prywatność i bezpieczeństwo
6. Praca z Copilotem w projekcie ML krok po kroku
7. Podsumowanie

**Cel:** nauczyć się korzystać z asystentów takich jak **GitHub Copilot** tak, by przyspieszać pracę, a nie wprowadzać do analizy trudne do zauważenia błędy.

## 1. Rola asystenta AI w pracy analityka

Asystent AI potrafi w kilka sekund napisać kod wczytania danych, wykres, `Pipeline` w scikit-learn czy test. **Nie wie jednak**, jaki jest cel biznesowy, skąd pochodzą dane ani co znaczy poprawny wynik. Generuje kod, który **wygląda** poprawnie, bo jest statystycznie prawdopodobny - również wtedy, gdy zawiera błąd.

**Zasada nadrzędna:** AI przyspiesza pisanie, ale **odpowiedzialność za wynik analizy zawsze zostaje po stronie analityka**. Kod, którego nie rozumiesz i nie sprawdziłeś, nie nadaje się do użycia w analizie.

W GitHub Copilot (w VS Code, także w notatnikach Jupyter) zwykle korzysta się z kilku trybów pracy:

| Tryb | Do czego służy | Przykład w Data Science |
|---|---|---|
| **Podpowiedzi w linii** | dokańczanie kodu w trakcie pisania | dokończenie `plt.` / `df.groupby(...)`, powtarzalny kod |
| **Czat (pytania)** | wyjaśnianie, pomysły, debugowanie | „Co oznacza ten błąd?”, „Wyjaśnij ten kod linia po linii” |
| **Edycja zaznaczonego kodu** | modyfikacja wskazanego fragmentu | „Przepisz tę pętlę na operację wektorową” |
| **Tryb agenta** | wieloetapowe zadania w projekcie (tworzenie plików, uruchamianie poleceń) | „Dodaj notatnik z baseline i testami” - **wymaga szczególnie uważnego przeglądu zmian** |

Nazwy i dostępne funkcje narzędzia zmieniają się często, dlatego liczą się zasady, a nie konkretne przyciski.

## 2. Jak pisać prompty w projektach Data Science

Dobry prompt to **dobra specyfikacja zadania**. Model nie widzi Twoich danych (chyba że ich fragment podasz), więc brakujące informacje zgaduje.

**Co zawierać w prompcie:**
1. **Cel i problem** - klasyfikacja czy regresja, co przewidujemy.
2. **Dane** - nazwy i typy kolumn, rozmiar, brakujące wartości; najlepiej kilka przykładowych wierszy (bez danych wrażliwych).
3. **Zmienna docelowa i metryka** - np. „`churn`, ocenianie AUC, klasy niezbalansowane”.
4. **Ograniczenia** - biblioteki i wersje (np. „pandas 3, scikit-learn 1.9”).
5. **Format odpowiedzi** - „krótki kod z komentarzami”, „sam kod bez wyjaśnień”.
6. **Kontekst** - zaznaczony kod, otwarte pliki lub odwołania do plików w czacie.

**Słaby prompt:**
> Zrób model do danych.

**Dobry prompt:**
> Mam plik `churn.csv` (3333 wiersze). Zmienna docelowa: `churn` (0/1, ok. 14% jedynek). Kolumny `state`, `area code`, `phone number` pomiń. Zbuduj `Pipeline` w scikit-learn (skalowanie + `LogisticRegression`), oceń go walidacją krzyżową (5 foldów, stratyfikowaną) metryką AUC i porównaj z `DummyClassifier`. Cała transformacja ma być wewnątrz `Pipeline` (bez wycieku danych). Użyj `random_state=42`.

**Dobre nawyki:**
- zadania dziel na **małe kroki** (wczytanie → EDA → baseline → model → ocena) zamiast jednego „zrób wszystko”,
- poproś o **wyjaśnienie** kodu, którego nie rozumiesz, i o **alternatywy**,
- gdy wynik jest błędny, wklej **komunikat błędu** i fragment kodu, a nie tylko „nie działa”,
- zapisuj sprawdzone prompty (np. w pliku w repozytorium).

## 3. Typowe pułapki kodu wygenerowanego przez AI



- **Brak powtarzalności:** brak `random_state`, losowe podziały bez ziarna - wynik zmienia się przy każdym uruchomieniu.
- **Ścieżki i dane na sztywno:** `C:/Users/jan/...` zamiast ścieżek względnych.
- **Nadmiernie skomplikowany kod:** pętle zamiast operacji wektorowych, niepotrzebne klasy i abstrakcje.
- **Ukryte założenia:** np. usunięcie wierszy z brakami (`dropna`) bez informacji, ile danych zniknęło.
- **Pewny siebie, ale błędny opis:** wyjaśnienia statystyczne (np. interpretacja p-wartości) bywają nieprecyzyjne - weryfikuj w literaturze.

## 4. Weryfikacja: lista kontrolna

Przed zaufaniem kodowi lub wynikowi od AI:

1. **Rozumiem każdą linię?** Jeśli nie, poproś o wyjaśnienie lub uprość.
2. **Dane:** liczba wierszy i kolumn, typy, braki, duplikaty, rozkład zmiennej docelowej zgadzają się z oczekiwaniami.
3. **Podział i wyciek:** podział na zbiory jest przed jakimkolwiek `fit`; kroki uczące się wchodzą do `Pipeline`.
4. **Baseline:** model jest lepszy od prostego punktu odniesienia.
5. **Metryka:** jest dobrana do problemu i policzona na danych, których model nie widział.
6. **Sensowność:** wyniki są wiarygodne (zbyt dobry wynik to często wyciek), a wnioski wynikają z danych.
7. **Powtarzalność:** ziarna losowości, wersje bibliotek, kod uruchamia się od zera (*Restart & Run All*).
8. **Testy:** proste `assert` na kluczowe założenia (kształty, zakresy wartości, brak `NaN` po przetworzeniu).

Poniżej mała funkcja kontrolna, o którą warto poprosić asystenta na początku każdej analizy i uruchamiać po każdym większym kroku przetwarzania.

In [2]:
from sklearn.datasets import load_breast_cancer

def basic_checks(df, target):
    print('Kształt:', df.shape)
    print('Brakujące wartości (łącznie):', int(df.isna().sum().sum()))
    print('Zduplikowane wiersze:', int(df.duplicated().sum()))
    print('Rozkład zmiennej docelowej:', df[target].value_counts(normalize=True).round(3).to_dict())
    assert df[target].notna().all(), 'brak wartości w zmiennej docelowej'

data = load_breast_cancer(as_frame=True).frame
basic_checks(data, 'target')

Kształt: (569, 31)
Brakujące wartości (łącznie): 0
Zduplikowane wiersze: 0
Rozkład zmiennej docelowej: {1: 0.627, 0: 0.373}


## 5. Prywatność i bezpieczeństwo

- **Nie wklejaj do promptów danych osobowych, poufnych danych firmy, haseł ani kluczy API.** Do opisu danych wystarczą nazwy kolumn, typy i zanonimizowane lub wymyślone przykłady.
- Klucze i hasła trzymaj w **zmiennych środowiskowych** lub pliku `.env` dodanym do `.gitignore`, a nie w notatniku. Notatniki trafiają do repozytoriów razem z wynikami komórek - **wyczyść wyjścia** zawierające dane wrażliwe przed zatwierdzeniem zmian.
- Sprawdź **politykę organizacji** wobec narzędzi AI (które usługi wolno używać, czy dane mogą opuszczać firmę, czy można wykluczyć wybrane pliki z kontekstu).
- **Zależności:** nie instaluj bezmyślnie bibliotek zasugerowanych przez AI; sprawdź, czy pakiet istnieje, jest utrzymywany i ma sensowną nazwę (zdarzają się zmyślone nazwy pakietów).
- **Agent z dostępem do terminala:** przeglądaj proponowane polecenia i zmiany plików, zanim je zatwierdzisz (szczególnie polecenia usuwające lub nadpisujące dane).

## 6. Praca z Copilotem w projekcie ML krok po kroku

| Etap projektu | Co można powierzyć AI | Co robisz samodzielnie |
|---|---|---|
| **Zrozumienie problemu** | burza mózgów: cechy, ryzyka, metryki | definicja celu, ograniczenia biznesowe |
| **Dane i EDA** | kod wykresów i statystyk, szkic czyszczenia | ocena jakości danych, decyzje o brakach i odstających |
| **Przygotowanie cech** | szkielet `ColumnTransformer` / `Pipeline` | kontrola wycieku, sens biznesowy cech |
| **Model** | baseline, porównanie modeli, `GridSearchCV` | wybór metryki, interpretacja wyników |
| **Ocena i interpretacja** | kod macierzy pomyłek, krzywych, SHAP | wnioski, ocena ryzyka i uczciwości modelu |
| **Wdrożenie** | szkielet aplikacji, zapis modelu, testy | bezpieczeństwo, monitorowanie, odpowiedzialność |
| **Dokumentacja** | szkice opisów, komentarzy, README | sprawdzenie zgodności opisu z faktami |

### Instrukcje projektu dla Copilota
Żeby nie powtarzać w każdym prompcie tych samych zasad, można je zapisać w pliku z instrukcjami dla projektu (w repozytorium: `.github/copilot-instructions.md`). Asystent dołącza go do kontekstu. Przykład dla projektu Data Science:

```markdown
# Zasady projektu

- Język: Python 3.13, pandas 3, scikit-learn 1.9. Nie używaj przestarzałych funkcji (np. DataFrame.append).
- Zawsze ustawiaj random_state=42 i ścieżki względne.
- Kroki uczące się z danych (skalowanie, imputacja, selekcja cech) umieszczaj w Pipeline; podział na zbiory robisz przed jakimkolwiek fit.
- Do każdego modelu dodaj baseline (DummyClassifier/DummyRegressor) i metrykę właściwą dla problemu (przy niezbalansowanych klasach nie samą accuracy).
- Wykresy: matplotlib, opisane osie, tytuły i legendy po polsku.
- Po złączeniach tabel sprawdzaj liczbę wierszy (validate= w merge).
- Nie wczytuj ani nie wypisuj danych osobowych.
```

**Szybki rytm pracy:** opisz małe zadanie → przeczytaj wygenerowany kod → uruchom go → sprawdź wynik (liczby, wykres) względem oczekiwań → dopiero wtedy przejdź dalej.

## 7. Podsumowanie

- Asystent AI **przyspiesza** pracę, ale **nie zwalnia z odpowiedzialności**: kod i wnioski weryfikuje analityk.
- Dobry prompt to **specyfikacja**: cel, dane, metryka, ograniczenia, wersje bibliotek, format odpowiedzi; zadania dziel na małe kroki.
- Najgroźniejsze błędy są **ciche**: wyciek danych, brak baseline, zwielokrotnione wiersze po złączeniu, usunięte funkcje, brak powtarzalności.
- Używaj **listy kontrolnej**: dane, podział, `Pipeline`, baseline, metryka, sensowność, powtarzalność, testy (`assert`).
- Chroń **prywatność**: żadnych danych osobowych ani sekretów w promptach i notatnikach.
- Zasady projektu zapisz w **pliku instrukcji**, a Copilota traktuj jak szybkiego, ale niedoświadczonego współpracownika, którego pracę trzeba przejrzeć.